# Data encoding: the same photos in five formats

The photo service stores and sends photo records: the photo, its user, and its camera. This
notebook writes the same 40,000 records in five encodings and compares them:

- **CSV**: plain text, one flat line for each record;
- **JSON**: semi-structured text; each record repeats all field names;
- **Avro** and **Protocol Buffers**: binary encodings with a schema; the schema has the names
  and types, so the data has only the values;
- **Parquet**: a columnar, compressed file format for analytics.

## Setup: the photo records

This cell only prepares the data; the encodings start below. It makes the photos as nested
records (like documents), with the user and the camera inside. It also generates the Python
code for the Protocol Buffers schema `photo.proto` (into `out/`).

In [1]:
import csv
import gzip
import io
import json
import sys
import time
from pathlib import Path

import fastavro
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from grpc_tools import protoc

import photo_data

OUT = Path("out")
OUT.mkdir(exist_ok=True)

# Generate the Python code for photo.proto, then import it.
protoc.main(["protoc", "-I.", f"--python_out={OUT}", "photo.proto"])
sys.path.insert(0, str(OUT))
import photo_pb2  # noqa: E402  (generated from photo.proto)


def photo_records() -> list[dict]:
    """The photos as nested records, with the user and the camera inside."""
    users = photo_data.users().set_index("user_id").account_name
    cameras = photo_data.cameras().set_index("camera_id")
    records = []
    for p in photo_data.photos().itertuples():
        camera = None
        if not pd.isna(p.camera_id):  # some photos have no camera data
            c = cameras.loc[int(p.camera_id)]
            camera = {
                "manufacturer": c.manufacturer,
                "print_name": c.print_name,
                "settings": p.camera_setting,
            }
        records.append(
            {
                "photo_id": p.photo_id,
                "path": p.path,
                "upload_date": int(p.upload_date.timestamp() * 1000),  # ms since 1970
                "size": p.size,
                "format": p.format,
                "title": p.title,
                "objects": p.objects.split(";"),
                "user": {"user_id": p.user_id, "account_name": users[p.user_id]},
                "camera": camera,
            }
        )
    return records


photos = photo_records()
one = next(r for r in photos if r["photo_id"] == 133422131)  # one photo, for the examples
len(photos)

40000

The photo that the examples use:

In [2]:
print(json.dumps(one, indent=2, ensure_ascii=False))

{
  "photo_id": 133422131,
  "path": "/st/u211/1U6uFl47Fy.jpg",
  "upload_date": 1638523112124,
  "size": 5.7,
  "format": "jpg",
  "title": "",
  "objects": [
    "person",
    "christmas tree"
  ],
  "user": {
    "user_id": 54351,
    "account_name": "ckaestne"
  },
  "camera": {
    "manufacturer": "Google",
    "print_name": "Google Pixel 5",
    "settings": "ƒ/1.8; 1/120; 4.44mm; ISO271"
  }
}


---

## 1. The five encodings

For each format, a function writes a list of records as bytes, and a function reads the bytes
back.

### CSV: one flat line for each record

CSV has no nested values. The user and the camera become columns, and the list of objects
becomes one string.

In [3]:
def flat(r: dict) -> dict:
    """A nested record as one flat row."""
    camera = r["camera"] or {}
    nested = ("user", "camera", "objects")
    return {k: v for k, v in r.items() if k not in nested} | {
        "objects": ";".join(r["objects"]),
        "user_id": r["user"]["user_id"],
        "account_name": r["user"]["account_name"],
        "camera": camera.get("print_name", ""),
        "camera_settings": camera.get("settings", ""),
    }


def write_csv(records: list[dict]) -> bytes:
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=list(flat(records[0])))
    writer.writeheader()
    writer.writerows(flat(r) for r in records)
    return buffer.getvalue().encode()


def read_csv(data: bytes) -> list[dict]:
    return list(csv.DictReader(io.StringIO(data.decode())))


print(write_csv([one]).decode())

photo_id,path,upload_date,size,format,title,objects,user_id,account_name,camera,camera_settings
133422131,/st/u211/1U6uFl47Fy.jpg,1638523112124,5.7,jpg,,person;christmas tree,54351,ckaestne,Google Pixel 5,ƒ/1.8; 1/120; 4.44mm; ISO271



### JSON: one object for each record (JSON Lines)

JSON keeps the nested values, but each record repeats all field names.

In [4]:
def write_json(records: list[dict]) -> bytes:
    return "".join(json.dumps(r) + "\n" for r in records).encode()


def read_json(data: bytes) -> list[dict]:
    return [json.loads(line) for line in data.decode().splitlines()]


print(write_json([one]).decode())

{"photo_id": 133422131, "path": "/st/u211/1U6uFl47Fy.jpg", "upload_date": 1638523112124, "size": 5.7, "format": "jpg", "title": "", "objects": ["person", "christmas tree"], "user": {"user_id": 54351, "account_name": "ckaestne"}, "camera": {"manufacturer": "Google", "print_name": "Google Pixel 5", "settings": "\u0192/1.8; 1/120; 4.44mm; ISO271"}}



### Avro: binary, with a schema

The schema `photo.avsc` has the names and types of the fields:

In [5]:
print(Path("photo.avsc").read_text())

{
  "type": "record",
  "name": "Photo",
  "fields": [
    {"name": "photo_id", "type": "long"},
    {"name": "path", "type": "string"},
    {"name": "upload_date", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "size", "type": "float"},
    {"name": "format", "type": "string"},
    {"name": "title", "type": "string"},
    {"name": "objects", "type": {"type": "array", "items": "string"}},
    {"name": "user", "type": {
      "type": "record", "name": "User",
      "fields": [{"name": "user_id", "type": "long"}, {"name": "account_name", "type": "string"}]}},
    {"name": "camera", "type": ["null", {
      "type": "record", "name": "Camera",
      "fields": [
        {"name": "manufacturer", "type": "string"},
        {"name": "print_name", "type": "string"},
        {"name": "settings", "type": "string"}]}]}
  ]
}



The encoded record has only the values, without names. An Avro *file* starts with a header
that has the schema, so that a reader can decode the values.

In [6]:
AVRO_SCHEMA = fastavro.parse_schema(json.loads(Path("photo.avsc").read_text()))


def write_avro(records: list[dict]) -> bytes:
    buffer = io.BytesIO()
    fastavro.writer(buffer, AVRO_SCHEMA, records)  # a file: the header, then the records
    return buffer.getvalue()


def read_avro(data: bytes) -> list[dict]:
    return list(fastavro.reader(io.BytesIO(data)))


def avro_message(record: dict) -> bytes:
    """One record without the header, for example as a message (the reader knows the schema)."""
    buffer = io.BytesIO()
    fastavro.schemaless_writer(buffer, AVRO_SCHEMA, record)
    return buffer.getvalue()


print(avro_message(one))

b'\xe6\xf0\x9e\x7f./st/u211/1U6uFl47Fy.jpg\xf8\xaa\xb4\xf9\xaf_ff\xb6@\x06jpg\x00\x04\x0cperson\x1cchristmas tree\x00\x9e\xd1\x06\x10ckaestne\x02\x0cGoogle\x1cGoogle Pixel 5:\xc6\x92/1.8; 1/120; 4.44mm; ISO271'


### Protocol Buffers: binary, with a schema

The schema `photo.proto` gives each field a number. The encoded message has the numbers and
the values, not the names.

In [7]:
print(Path("photo.proto").read_text())

syntax = "proto3";

message Photo {
  int64 photo_id = 1;
  string path = 2;
  int64 upload_date = 3;  // milliseconds since 1970
  float size = 4;
  string format = 5;
  string title = 6;
  repeated string objects = 7;
  User user = 8;
  optional Camera camera = 9;
}

message User {
  int64 user_id = 1;
  string account_name = 2;
}

message Camera {
  string manufacturer = 1;
  string print_name = 2;
  string settings = 3;
}

message PhotoList {
  repeated Photo photos = 1;
}



In [8]:
def to_proto(r: dict) -> photo_pb2.Photo:
    """A record as a Photo message (the class is generated from photo.proto)."""
    message = photo_pb2.Photo(**{k: v for k, v in r.items() if k not in ("user", "camera")})
    message.user.CopyFrom(photo_pb2.User(**r["user"]))
    if r["camera"]:
        message.camera.CopyFrom(photo_pb2.Camera(**r["camera"]))
    return message


def write_protobuf(records: list[dict]) -> bytes:
    return photo_pb2.PhotoList(photos=[to_proto(r) for r in records]).SerializeToString()


def read_protobuf(data: bytes):
    return photo_pb2.PhotoList.FromString(data).photos


print(to_proto(one).SerializeToString())

b'\x08\xb3\xb8\xcf?\x12\x17/st/u211/1U6uFl47Fy.jpg\x18\xbc\x95\xda\xfc\xd7/%ff\xb6@*\x03jpg:\x06person:\x0echristmas treeB\x0e\x08\xcf\xa8\x03\x12\x08ckaestneJ7\n\x06Google\x12\x0eGoogle Pixel 5\x1a\x1d\xc6\x92/1.8; 1/120; 4.44mm; ISO271'


### Parquet: columns, not rows

Parquet stores the values of each column together, and compresses them. The schema is in the
file. Here it is an Arrow schema with nested columns:

In [9]:
ARROW_SCHEMA = pa.schema(
    [
        ("photo_id", pa.int64()),
        ("path", pa.string()),
        ("upload_date", pa.timestamp("ms", tz="UTC")),
        ("size", pa.float32()),
        ("format", pa.string()),
        ("title", pa.string()),
        ("objects", pa.list_(pa.string())),
        ("user", pa.struct([("user_id", pa.int64()), ("account_name", pa.string())])),
        (
            "camera",
            pa.struct(
                [
                    ("manufacturer", pa.string()),
                    ("print_name", pa.string()),
                    ("settings", pa.string()),
                ]
            ),
        ),
    ]
)


def write_parquet(records: list[dict]) -> bytes:
    buffer = io.BytesIO()
    pq.write_table(pa.Table.from_pylist(records, schema=ARROW_SCHEMA), buffer)
    return buffer.getvalue()


def read_parquet(data: bytes) -> pa.Table:
    return pq.read_table(io.BytesIO(data))


read_parquet(write_parquet(photos[:3])).to_pandas()

,photo_id,path,upload_date,size,format,title,objects,user,camera
0,133388021,/st/x171/pSWmM3D4I4.heic,2021-07-01 00:40:24.929000+00:00,1.5,heic,Thanksgiving,"[tree, food]","{'user_id': 28926, 'account_name': 'lena.jensen'}","{'manufacturer': 'Apple', 'print_name': 'Apple..."
1,133388022,/st/yfef/VnKl9cW2Fa.jpg,2021-07-01 00:41:46.717000+00:00,3.2,jpg,,[sunset],"{'user_id': 28926, 'account_name': 'lena.jensen'}","{'manufacturer': 'Apple', 'print_name': 'Apple..."
2,133388023,/st/u1ff/WLlm92dDcn.heic,2021-07-01 00:44:15.660000+00:00,1.9,heic,Thanksgiving,"[person, food]","{'user_id': 28926, 'account_name': 'lena.jensen'}","{'manufacturer': 'Apple', 'print_name': 'Apple..."


### All formats, and how to read only some columns

Parquet can read only the columns that a program needs. The other formats must parse the
whole records to get some of the columns.

In [10]:
FORMATS = {
    "CSV": (write_csv, read_csv),
    "JSON": (write_json, read_json),
    "Avro": (write_avro, read_avro),
    "Protobuf": (write_protobuf, read_protobuf),
    "Parquet": (write_parquet, read_parquet),
}


def read_columns(fmt: str, data: bytes, columns: list[str]):
    if fmt == "Parquet":
        return pq.read_table(io.BytesIO(data), columns=columns)  # reads only these columns

    _, read = FORMATS[fmt]
    records = read(data)  # parse all records, then take the columns
    if fmt == "Protobuf":
        return [[getattr(r, c) for c in columns] for r in records]
    return [[r[c] for c in columns] for r in records]

---

## 2. One photo as one message

The size of one record, for example as a message in a queue:

In [11]:
pd.DataFrame(
    {
        "bytes": {
            "CSV (one line, without the header)": len(write_csv([one]).splitlines()[1]),
            "JSON": len(json.dumps(one).encode()),
            "Avro": len(avro_message(one)),
            "Protobuf": len(to_proto(one).SerializeToString()),
        }
    }
)

,bytes
"CSV (one line, without the header)",138
JSON,347
Avro,132
Protobuf,144


JSON is the largest, because it has all field names. Avro and Protobuf write only the values:
the schema has the names and types.

As *files*, Avro and Parquet have extra data. For only one photo, this is large:

In [12]:
print(f"Avro file with one photo:    {len(write_avro([one])):>5} bytes (header with the schema)")
print(f"Parquet file with one photo: {len(write_parquet([one])):>5} bytes (metadata)")

Avro file with one photo:      956 bytes (header with the schema)
Parquet file with one photo:  3919 bytes (metadata)


These file formats are for many records.

---

## 3. All 40,000 photos: size and speed

The times are the best of 3 runs on one machine. They change a little from run to run.
The files are written to `out/`.

In [13]:
def best_of_3(f) -> float:
    """The shortest time of 3 runs of f(), in seconds."""
    times = []
    for _ in range(3):
        start = time.perf_counter()
        f()
        times.append(time.perf_counter() - start)
    return min(times)


def measure(fmt: str) -> dict:
    """Write all photos in one format, save the file, and measure the size and the times."""
    write, read = FORMATS[fmt]
    data = write(photos)
    (OUT / f"photos.{fmt.lower()}").write_bytes(data)

    if fmt == "Parquet":
        gzipped = "(compressed)"  # Parquet is compressed already
    else:
        gzipped = f"{len(gzip.compress(data)) / 1e6:.2f} MB"

    two_columns = best_of_3(lambda: read_columns(fmt, data, ["photo_id", "size"]))
    return {
        "size": f"{len(data) / 1e6:.2f} MB",
        "gzip": gzipped,
        "write": f"{best_of_3(lambda: write(photos)):.2f} s",
        "read": f"{best_of_3(lambda: read(data)):.2f} s",
        "read 2 columns": f"{two_columns:.3f} s",
    }


results = {fmt: measure(fmt) for fmt in FORMATS}
pd.DataFrame(results).T

,size,gzip,write,read,read 2 columns
CSV,5.38 MB,1.43 MB,0.19 s,0.12 s,0.150 s
JSON,13.68 MB,1.61 MB,0.13 s,0.19 s,0.209 s
Avro,5.05 MB,1.49 MB,0.29 s,0.25 s,0.269 s
Protobuf,5.65 MB,1.61 MB,0.23 s,0.01 s,0.024 s
Parquet,1.92 MB,(compressed),0.09 s,0.01 s,0.002 s


JSON is more than twice as large as the others; general compression (gzip) removes most of the
repeated names. Parquet is the smallest (a column has similar values, which compress well),
and it reads 2 columns very fast, because it does not read the other columns. The Protobuf
library parses in C, so it is fast to read.

---

## 4. Types: what comes back, and a wrong value

CSV has no types. It gives back only strings:

In [14]:
back = read_csv(write_csv([one]))[0]
print(f"size = {back['size']!r}")
print(f"objects = {back['objects']!r}")

size = '5.7'
objects = 'person;christmas tree'


A record with a wrong value: `size = '5.7 MB'` (text, not a number). The formats with a schema
reject it when they write it:

In [15]:
bad = one | {"size": "5.7 MB"}

for fmt, (write, _) in FORMATS.items():
    try:
        write([bad])
        print(f"{fmt:<9} accepted")
    except Exception as e:
        error = str(e).splitlines()[0][:70]
        print(f"{fmt:<9} rejected: {type(e).__name__}: {error}")

CSV       accepted
JSON      accepted
Avro      rejected: ValueError: could not convert string to float: '5.7 MB'
Protobuf  rejected: TypeError: must be real number, not str
Parquet   rejected: ArrowInvalid: Could not convert '5.7 MB' with type str: tried to convert to float32
